In [23]:
from pyspark.sql import SparkSession 
import pyspark.sql.functions as F

In [24]:
spark = (
    SparkSession.builder 
    .appName("aula_03_transformacoes")
    .getOrCreate()
)

In [25]:
df = spark.read.csv("/workspace/datasets/03_transformacoes/clientes.csv", header= True, inferSchema= True)
df.show()

+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|      nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|         1| Cliente 1|   32|Feira de Santana|    BA|   Engenheiro|    12925.67|   M|false|   2025-02-22|
|         2| Cliente 2|   32|Feira de Santana|    BA|   Engenheiro|     4852.23|   M|false|   2024-01-16|
|         3| Cliente 3|   29|Feira de Santana|    BA|   Engenheiro|      8919.6|   F|false|   2023-02-08|
|         4| Cliente 4|   64|        Salvador|    BA|     Analista|     4253.53|   F|false|   2023-01-03|
|         5| Cliente 5|   60|        Salvador|    BA|   Engenheiro|     2710.92|   M|false|   2023-09-15|
|         6| Cliente 6|   30|        Salvador|    BA|   Engenheiro|    10282.08|   F|false|   2023-01-29|
|         7| Cliente 7|   46|Feira de Santana|

In [26]:
df.printSchema()

root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- profissao: string (nullable = true)
 |-- renda_mensal: double (nullable = true)
 |-- sexo: string (nullable = true)
 |-- ativo: boolean (nullable = true)
 |-- data_cadastro: date (nullable = true)



In [27]:
df_clientes = (
    df.select("cliente_id", "nome", "cidade", "renda_mensal")
)

In [28]:
df_clientes.explain(True)

== Parsed Logical Plan ==
'Project ['cliente_id, 'nome, 'cidade, 'renda_mensal]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, cidade: string, renda_mensal: double
Project [cliente_id#326, nome#327, cidade#329, renda_mensal#332]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Optimized Logical Plan ==
Project [cliente_id#326, nome#327, cidade#329, renda_mensal#332]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Physical Plan ==
FileScan csv [cliente_id#326,nome#327,cidade#329,renda_mensal#332] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/03_transformacoes/clientes.csv], Partitio

In [29]:
df_sem_dados_sensiveis = (
    df.drop("sexo", "estado")
)

In [30]:
df_sem_dados_sensiveis.explain(True)

== Parsed Logical Plan ==
Project [cliente_id#326, nome#327, idade#328, cidade#329, profissao#331, renda_mensal#332, ativo#334, data_cadastro#335]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, profissao: string, renda_mensal: double, ativo: boolean, data_cadastro: date
Project [cliente_id#326, nome#327, idade#328, cidade#329, profissao#331, renda_mensal#332, ativo#334, data_cadastro#335]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Optimized Logical Plan ==
Project [cliente_id#326, nome#327, idade#328, cidade#329, profissao#331, renda_mensal#332, ativo#334, data_cadastro#335]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro

In [31]:
df_tratado = (
    df.withColumn("renda_anual", (F.col("renda_mensal") * 12))
    .withColumn("idade", (F.col("idade") + 5))
    .withColumn(
        "maior_idade",
        F.when(F.col("idade") >= 18, "Sim")
        .otherwise("Não")
    )
)


In [32]:
df_tratado.show(10)

+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+------------------+-----------+
|cliente_id|      nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|       renda_anual|maior_idade|
+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+------------------+-----------+
|         1| Cliente 1|   37|Feira de Santana|    BA|   Engenheiro|    12925.67|   M|false|   2025-02-22|         155108.04|        Sim|
|         2| Cliente 2|   37|Feira de Santana|    BA|   Engenheiro|     4852.23|   M|false|   2024-01-16|58226.759999999995|        Sim|
|         3| Cliente 3|   34|Feira de Santana|    BA|   Engenheiro|      8919.6|   F|false|   2023-02-08|107035.20000000001|        Sim|
|         4| Cliente 4|   69|        Salvador|    BA|     Analista|     4253.53|   F|false|   2023-01-03|          51042.36|        Sim|
|         5| Cliente 5|   65|        Salv

In [33]:
df_tratado.explain(True)

== Parsed Logical Plan ==
'Project [cliente_id#326, nome#327, idade#422, cidade#329, estado#330, profissao#331, renda_mensal#332, sexo#333, ativo#334, data_cadastro#335, renda_anual#410, CASE WHEN ('idade >= 18) THEN Sim ELSE Não END AS maior_idade#434]
+- Project [cliente_id#326, nome#327, (idade#328 + 5) AS idade#422, cidade#329, estado#330, profissao#331, renda_mensal#332, sexo#333, ativo#334, data_cadastro#335, renda_anual#410]
   +- Project [cliente_id#326, nome#327, idade#328, cidade#329, estado#330, profissao#331, renda_mensal#332, sexo#333, ativo#334, data_cadastro#335, (renda_mensal#332 * cast(12 as double)) AS renda_anual#410]
      +- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date, renda_anual

In [34]:
df_renomeado = (
    df.withColumnRenamed("cliente_id", "id")
    .withColumnRenamed("nome", "cliente")
    .withColumnRenamed("renda_mensal", "salario")
)

In [35]:
df_renomeado.explain(True)

== Parsed Logical Plan ==
Project [id#506, cliente#517, idade#328, cidade#329, estado#330, profissao#331, renda_mensal#332 AS salario#528, sexo#333, ativo#334, data_cadastro#335]
+- Project [id#506, nome#327 AS cliente#517, idade#328, cidade#329, estado#330, profissao#331, renda_mensal#332, sexo#333, ativo#334, data_cadastro#335]
   +- Project [cliente_id#326 AS id#506, nome#327, idade#328, cidade#329, estado#330, profissao#331, renda_mensal#332, sexo#333, ativo#334, data_cadastro#335]
      +- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Analyzed Logical Plan ==
id: int, cliente: string, idade: int, cidade: string, estado: string, profissao: string, salario: double, sexo: string, ativo: boolean, data_cadastro: date
Project [id#506, cliente#517, idade#328, cidade#329, estado#330, profissao#331, renda_mensal#332 AS salario#528, sexo#333, ativo#334, data_cadastro#335]
+- Project [id#506, nom

In [36]:
df_alias = (
    df.select(
        F.col("nome").alias("cliente"),
        F.col("idade")
    )
)


In [ ]:
df_alias.show()

In [37]:
df_alias.explain(True)

== Parsed Logical Plan ==
'Project ['nome AS cliente#539, 'idade]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Analyzed Logical Plan ==
cliente: string, idade: int
Project [nome#327 AS cliente#539, idade#328]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Optimized Logical Plan ==
Project [nome#327 AS cliente#539, idade#328]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Physical Plan ==
*(1) Project [nome#327 AS cliente#539, idade#328]
+- FileScan csv [nome#327,idade#328] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/03_transformacoes/clientes.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<nome:string,idade:int>

In [38]:
df_rename = (
    df.withColumnRenamed("nome", "cliente")
)

In [ ]:
df_rename.show()

In [39]:
df_rename.explain(True)

== Parsed Logical Plan ==
Project [cliente_id#326, nome#327 AS cliente#542, idade#328, cidade#329, estado#330, profissao#331, renda_mensal#332, sexo#333, ativo#334, data_cadastro#335]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Analyzed Logical Plan ==
cliente_id: int, cliente: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Project [cliente_id#326, nome#327 AS cliente#542, idade#328, cidade#329, estado#330, profissao#331, renda_mensal#332, sexo#333, ativo#334, data_cadastro#335]
+- Relation [cliente_id#326,nome#327,idade#328,cidade#329,estado#330,profissao#331,renda_mensal#332,sexo#333,ativo#334,data_cadastro#335] csv

== Optimized Logical Plan ==
Project [cliente_id#326, nome#327 AS cliente#542, idade#328, cidade#329, estado#330, profissao#331, renda_mensal#332, sexo#333, ativo#334, data_ca

In [ ]:
spark.stop()